# 2023 학점(A/CD/F) 비율 + 졸업자 취업통계 — P2-G3 시계열 확장용 2023 표본

이 노트북 **하나만** 사용해 2023년 데이터를 처리한다(다른 스크립트/노트북 생성 금지).

- 목표 스키마: `/home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_3/runner/P2_G3_FINAL_BLUEPRINT.md`의
  `D02 dept_outcomes_2024.parquet`와 동일한 grain(`analysis_year × school × campus × dept`)·컬럼명·dtype을
  최대한 맞춰서, 향후 2024 mart와 `analysis_year`만 다르게 시계열로 이어붙일 수 있게 한다.
- 원천:
  1. `전공과목 성적 분포 (대학)_2023-07-1220531571.xlsx` → A/CD/F 비율
  2. `2023년 학교별 학과별 고등교육기관 졸업자 취업통계_241230 (1).xlsx` → 졸업자 취업통계(취업률)
- P3 허용/금지 경계(블루프린트 §5)를 그대로 따른다: 결측 대체·스케일러·원핫·PCA·이상치 제거 금지,
  raw/std 동시 보존, 품질 플래그·provenance 기록은 허용.


In [1]:
import hashlib
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_DIR = Path("/home/sieg/projects-wsl/SBS_dataScience")
P2_4_DIR = PROJECT_DIR / "workbook/p2/p2_4"
P2_3_DIR = PROJECT_DIR / "workbook/p2/p2_3"

GRADE_XLSX_PATH = P2_4_DIR / "전공과목 성적 분포 (대학)_2023-07-1220531571.xlsx"
EMPLOYMENT_XLSX_PATH = P2_4_DIR / "2023년 학교별 학과별 고등교육기관 졸업자 취업통계_241230 (1).xlsx"

REFERENCE_D02_PATH = P2_3_DIR / "p3_1/dept_outcomes_2024.parquet"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 200)

print("GRADE_XLSX_PATH exists:", GRADE_XLSX_PATH.exists())
print("EMPLOYMENT_XLSX_PATH exists:", EMPLOYMENT_XLSX_PATH.exists())
print("REFERENCE_D02_PATH exists:", REFERENCE_D02_PATH.exists())


GRADE_XLSX_PATH exists: True
EMPLOYMENT_XLSX_PATH exists: True
REFERENCE_D02_PATH exists: True


## 0. 목표 스키마 확인 — `dept_outcomes_2024.parquet`(D02) 그대로 읽어서 컬럼/타입 참조

새로 만들 것을 상상하지 않고, 실제 존재하는 2024 D02 파일의 컬럼명·dtype을 그대로 목표로 삼는다.

In [2]:
d02_2024 = pd.read_parquet(REFERENCE_D02_PATH)
print("D02 2024 shape:", d02_2024.shape)
d02_dtypes = d02_2024.dtypes
print(d02_dtypes)

D02_TARGET_COLUMNS = d02_dtypes.index.tolist()
D02_TARGET_DTYPES = d02_dtypes.to_dict()

school_name_std_universe_2024 = set(d02_2024["school_name_std"].dropna().unique())
print("\n2024 school_name_std 고유 수:", len(school_name_std_universe_2024))


D02 2024 shape: (10242, 32)
analysis_year                                 Int16
outcome_row_id                                  str
school_name_raw                              string
school_name_base_raw                         string
school_name_std                              string
campus_name_raw                              string
campus_seq                                   string
campus_branch                                string
campus_name_std                            category
dept_name_raw                                string
dept_name_std                                string
dept_field_raw                               string
dept_field_std                               string
credit_forfeit_flag                         boolean
selectivity_proxy_pct                       Float32
a_rate_pct                                  Float32
cd_rate_pct                                 Float32
f_rate_pct                                  Float32
employment_rate_pct                 

## 1. 전공과목 성적 분포 (대학) 로드 + 헤더 평탄화

원본은 3단 헤더(행3: 대분류, 행4: 등급군A+/A0/...F, 행5: 학생수/비율(%))다.
컬럼명을 하드코딩하지 않고 3개 헤더 행을 프로그램적으로 합쳐서 만든다.

In [3]:
_grade_header_raw = pd.read_excel(GRADE_XLSX_PATH, sheet_name="Sheet1", header=None, nrows=6)
row_main = _grade_header_raw.iloc[3]
row_grade_group = _grade_header_raw.iloc[4]
row_subcol = _grade_header_raw.iloc[5]

FIXED_FIELD_NAMES = [
    "기준연도", "학교명", "단과대학", "학과_전공", "구분", "학과특성", "학기",
    "전공과목_성적인정_총학생수", "평점_만점",
]
n_fixed = len(FIXED_FIELD_NAMES)

flat_columns = list(FIXED_FIELD_NAMES)
_cur_group = None
for i in range(n_fixed, len(row_main)):
    if pd.notna(row_grade_group[i]):
        _cur_group = str(row_grade_group[i]).strip()
    sub = row_subcol[i]
    sub_label = "학생수" if isinstance(sub, str) and "학생수" in sub else "비율"
    flat_columns.append(f"{_cur_group}_{sub_label}")

print(len(flat_columns))
print(flat_columns)


35
['기준연도', '학교명', '단과대학', '학과_전공', '구분', '학과특성', '학기', '전공과목_성적인정_총학생수', '평점_만점', 'A+_학생수', 'A+_비율', 'A0_학생수', 'A0_비율', 'A-_학생수', 'A-_비율', 'B+_학생수', 'B+_비율', 'B0_학생수', 'B0_비율', 'B-_학생수', 'B-_비율', 'C+_학생수', 'C+_비율', 'C0_학생수', 'C0_비율', 'C-_학생수', 'C-_비율', 'D+_학생수', 'D+_비율', 'D0_학생수', 'D0_비율', 'D-_학생수', 'D-_비율', 'F_학생수', 'F_비율']


In [4]:
grade_raw = pd.read_excel(GRADE_XLSX_PATH, sheet_name="Sheet1", header=None, skiprows=6)
grade_raw.columns = flat_columns
print("grade_raw shape:", grade_raw.shape)
grade_raw.head(3)


grade_raw shape: (22559, 35)


,기준연도,학교명,단과대학,학과_전공,구분,학과특성,학기,전공과목_성적인정_총학생수,평점_만점,A+_학생수,A+_비율,A0_학생수,A0_비율,A-_학생수,A-_비율,B+_학생수,B+_비율,B0_학생수,B0_비율,B-_학생수,B-_비율,C+_학생수,C+_비율,C0_학생수,C0_비율,C-_학생수,C-_비율,D+_학생수,D+_비율,D0_학생수,D0_비율,D-_학생수,D-_비율,F_학생수,F_비율
0,2022,가야대학교(김해),단과대구분없음,간호학과,주간,일반과정,1학기,4723,4.5,493,10.4,994,21.0,0,0.0,1202,25.4,739,15.6,0,0.0,653,13.8,352,7.5,0,0.0,129,2.7,79,1.7,0,0.0,82,1.7
1,2022,가야대학교(김해),단과대구분없음,간호학과,주간,일반과정,2학기,5105,4.5,582,11.4,1030,20.2,0,0.0,1436,28.1,728,14.3,0,0.0,697,13.7,362,7.1,0,0.0,132,2.6,89,1.7,0,0.0,49,1.0
2,2022,가야대학교(김해),단과대구분없음,경영물류학과,주간,일반과정,1학기,276,4.5,61,22.1,19,6.9,0,0.0,83,30.1,20,7.2,0,0.0,65,23.6,10,3.6,0,0.0,9,3.3,1,0.4,0,0.0,8,2.9


### 1.1 성적분포 EDA — 연도/구분/학기/학과특성 분포, 결측, 스키마 대조

**중요 발견**: 파일명·타이틀은 "2023년_"이지만 실제 `기준연도` 컬럼 값은 아래에서 확인한다.
대학알리미류 공시는 발행연도보다 1년 전 학사연도를 담는 경우가 흔하다 — 이 노트북은 그 값을
그대로 보존하고, `analysis_year`는 사용자가 지정한 2023(시계열 슬롯)으로 별도 배정한다.

In [5]:
print("기준연도 unique:", sorted(grade_raw["기준연도"].dropna().unique().tolist()))
print()
print("구분(주간/야간/원격) 분포:\n", grade_raw["구분"].value_counts(dropna=False))
print()
print("학기 분포:\n", grade_raw["학기"].value_counts(dropna=False))
print()
print("학과특성 분포:\n", grade_raw["학과특성"].value_counts(dropna=False))
print()
print("학교명 고유 수:", grade_raw["학교명"].nunique())
print("학과_전공 고유 수:", grade_raw["학과_전공"].nunique())
print()
print("컬럼별 결측률(%):")
print((grade_raw.isna().mean() * 100).round(2))


기준연도 unique: [2022]

구분(주간/야간/원격) 분포:
 구분
주간    20895
야간      896
원격      768
Name: count, dtype: int64

학기 분포:
 학기
1학기    11309
2학기    11165
3학기       77
4학기        8
Name: count, dtype: int64

학과특성 분포:
 학과특성
일반과정       22005
계약학과         404
산업체위탁        122
학석사통합과정       16
특별과정          12
Name: count, dtype: int64

학교명 고유 수: 244
학과_전공 고유 수: 5095

컬럼별 결측률(%):
기준연도              0.0
학교명               0.0
단과대학              0.0
학과_전공             0.0
구분                0.0
학과특성              0.0
학기                0.0
전공과목_성적인정_총학생수    0.0
평점_만점             0.0
A+_학생수            0.0
A+_비율             0.0
A0_학생수            0.0
A0_비율             0.0
A-_학생수            0.0
A-_비율             0.0
B+_학생수            0.0
B+_비율             0.0
B0_학생수            0.0
B0_비율             0.0
B-_학생수            0.0
B-_비율             0.0
C+_학생수            0.0
C+_비율             0.0
C0_학생수            0.0
C0_비율             0.0
C-_학생수            0.0
C-_비율             0.0
D+_학생수            0.0
D+_비율             

### 1.2 A/CD/F 비율 계산 — 2024 파이프라인(`scripts/build_p2_g1_concat.py`)과 동일한 필터·공식 재사용

`구분=="주간"`, `학기 in [1학기,2학기]` 필터와 `a=A+ +A0+A-`, `cd=C+ +C0+C-+D+ +D0+D-`, `f=F`,
분모=`전공과목_성적인정_총학생수` 공식을 그대로 적용한다. 새 규칙을 만들지 않는다.

In [6]:
import sys
sys.path.insert(0, str(PROJECT_DIR / "scripts"))
from build_p2_g1_concat import normalize_key, load_department_alias, label_department  # noqa: E402

dept_alias = load_department_alias()
print("dept_alias 항목 수:", len(dept_alias))

grade_f = grade_raw[
    grade_raw["구분"].eq("주간") & grade_raw["학기"].isin(["1학기", "2학기"])
].copy()
print("필터 후 행수:", len(grade_f), "/ 전체", len(grade_raw))

grade_f["학과_계열"] = label_department(grade_f, dept_alias, "학과_전공")

count_cols = [
    "전공과목_성적인정_총학생수",
    "A+_학생수", "A0_학생수", "A-_학생수",
    "C+_학생수", "C0_학생수", "C-_학생수",
    "D+_학생수", "D0_학생수", "D-_학생수",
    "F_학생수",
]
for c in count_cols:
    grade_f[c] = pd.to_numeric(grade_f[c], errors="coerce").fillna(0)

grade_f["a_count"] = grade_f["A+_학생수"] + grade_f["A0_학생수"] + grade_f["A-_학생수"]
grade_f["cd_count"] = (
    grade_f["C+_학생수"] + grade_f["C0_학생수"] + grade_f["C-_학생수"]
    + grade_f["D+_학생수"] + grade_f["D0_학생수"] + grade_f["D-_학생수"]
)
grade_f["f_count"] = grade_f["F_학생수"]

semester = (
    grade_f.groupby(["학교명", "학과_계열", "학기"], dropna=False)
    .agg(
        denominator=("전공과목_성적인정_총학생수", "sum"),
        a_count=("a_count", "sum"),
        cd_count=("cd_count", "sum"),
        f_count=("f_count", "sum"),
    )
    .reset_index()
)


def _pct(num, den):
    num_arr = num.to_numpy(dtype=float)
    den_arr = den.to_numpy(dtype=float)
    out = np.full(len(num_arr), np.nan, dtype=float)
    np.divide(num_arr * 100, den_arr, out=out, where=den_arr > 0)
    return pd.Series(out, index=num.index)


semester["a_rate_pct"] = _pct(semester["a_count"], semester["denominator"])
semester["cd_rate_pct"] = _pct(semester["cd_count"], semester["denominator"])
semester["f_rate_pct"] = _pct(semester["f_count"], semester["denominator"])

grade_2023 = (
    semester.groupby(["학교명", "학과_계열"], dropna=False)
    .agg(a_rate_pct=("a_rate_pct", "mean"), cd_rate_pct=("cd_rate_pct", "mean"), f_rate_pct=("f_rate_pct", "mean"))
    .reset_index()
)
print("학교x학과_계열 grain 행수:", grade_2023.shape)
grade_2023.head(5)


dept_alias 항목 수: 9980
필터 후 행수: 20810 / 전체 22559


학교x학과_계열 grain 행수: (9226, 5)


,학교명,학과_계열,a_rate_pct,cd_rate_pct,f_rate_pct
0,가야대학교(김해),간호,31.530556,25.378143,1.348014
1,가야대학교(김해),경영물류,28.941423,29.276878,2.970188
2,가야대학교(김해),경찰소방,28.516540,26.675937,3.798302
3,가야대학교(김해),귀금속주얼리,33.284600,24.642625,0.000000
4,가야대학교(김해),물리치료,25.798374,31.027107,5.218533


## 2. 2023년 학교별 학과별 고등교육기관 졸업자 취업통계 로드 + EDA

`학교별 학과별` 시트, 실제 리프 헤더는 15행(0-index 14)에 있다(위 그룹핑 행은 병합셀 시각용).

In [7]:
emp_raw = pd.read_excel(EMPLOYMENT_XLSX_PATH, sheet_name="학교별 학과별", header=14)
print("emp_raw shape:", emp_raw.shape)
print()
print("조사기준일 unique:", emp_raw["조사기준일"].unique())
print()
print("학제 분포:\n", emp_raw["학제"].value_counts())
print()
print("과정구분 분포:\n", emp_raw["과정구분"].value_counts())
print()
print("학위구분 분포(NaN=학부/전문대):\n", emp_raw["학위구분"].value_counts(dropna=False))
print()
print("학교상태 분포:\n", emp_raw["학교상태"].value_counts())
print()
print("본분교 분포:\n", emp_raw["본분교"].value_counts())
print()
print("'건강보험' 텍스트를 포함한 컬럼(존재 여부 확인):", [c for c in emp_raw.columns if "건강보험" in c])


emp_raw shape: (25839, 94)

조사기준일 unique: <ArrowStringArray>
['2023.12.31']
Length: 1, dtype: str

학제 분포:
 학제
대학교          8889
일반대학원        7928
전문대학         3683
특수대학원        3595
전문대학원         877
사이버대학(대학)     338
기능대학          183
교육대학          126
산업대학           68
전공대학           35
각종대학(대학)       33
방송통신대학         31
사이버대학(전문)      24
원격대학(전문)       14
원격대학(대학)        8
사내대학(전문)        3
사내대학(대학)        3
기술대학            1
Name: count, dtype: int64

과정구분 분포:
 과정구분
대학원과정     12400
대학과정       9497
전문대학과정     3942
Name: count, dtype: int64

학위구분 분포(NaN=학부/전문대):
 학위구분
NaN    13439
석사      8727
박사      3673
Name: count, dtype: int64

학교상태 분포:
 학교상태
기존        25362
폐교          350
학교명 변경      127
Name: count, dtype: int64

본분교 분포:
 본분교
본교      24427
캠퍼스       792
분교        557
캠퍼스1       46
캠퍼스2       17
Name: count, dtype: int64

'건강보험' 텍스트를 포함한 컬럼(존재 여부 확인): []


### 2.1 학부(4년제 대학과정) 스코프로 제한 + 취업률 계산

- `과정구분=="대학과정"`(학부 수준, 대학원·전문대학 제외) + `학교상태=="기존"`(폐교·교명변경 제외)
- `취업률_계`는 KEDI가 이미 계산한 공식 취업률(%)이므로 그대로 `employment_rate_pct`로 사용한다.
- **`health_employment_rate_pct`(건보가입취업률)는 이 파일에 원본 컬럼이 없어 산출 불가** — 임의로
  추정하지 않고 전부 결측(`NaN`)으로 남긴다(P3 금지 원칙: 결측 대체 금지).

In [8]:
emp_f = emp_raw[
    emp_raw["과정구분"].eq("대학과정") & emp_raw["학교상태"].eq("기존")
].copy()
print("필터 후 행수:", len(emp_f), "/ 전체", len(emp_raw))

emp_f["employment_rate_pct"] = pd.to_numeric(emp_f["취업률_계"], errors="coerce")
emp_f["health_employment_rate_pct"] = np.nan

emp_f["학과_계열"] = label_department(emp_f, dept_alias, "학과명")

employment_2023 = (
    emp_f.groupby(["학교명", "학과_계열"], dropna=False)
    .agg(
        employment_rate_pct=("employment_rate_pct", "mean"),
        graduates_n=("졸업자_계", "sum"),
        campus_status_raw=("본분교", lambda s: " | ".join(sorted(set(s.dropna().astype(str))))),
    )
    .reset_index()
)
employment_2023["health_employment_rate_pct"] = np.nan
print("학교x학과_계열 grain 행수:", employment_2023.shape)
employment_2023.head(5)


필터 후 행수: 9406 / 전체 25839


학교x학과_계열 grain 행수: (8360, 6)


,학교명,학과_계열,employment_rate_pct,graduates_n,campus_status_raw,health_employment_rate_pct
0,LH토지주택대학교,<NA>,100.0,17,본교,NaN
1,가야대학교,간호,85.9,177,본교,NaN
2,가야대학교,경영물류,70.4,30,본교,NaN
3,가야대학교,경찰소방,31.3,17,본교,NaN
4,가야대학교,귀금속주얼리,57.9,21,본교,NaN


## 3. 학교명 정규화 — D02와 동일한 컬럼 이름으로 (raw/base_raw/std, campus raw/std)

2024 D02의 실제 값 패턴(`school_name_raw="가야대학교(김해)"` → `school_name_base_raw`/`school_name_std`=
"가야대학교"`, `campus_name_raw`/`campus_name_std`="김해")을 그대로 재현한다. 새 규칙을 발명하지 않는다.

In [9]:
PAREN_PATTERN = re.compile(r"\(([^)]*)\)")


def normalize_text(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return ""
    text = unicodedata.normalize("NFKC", str(value))
    text = text.replace("​", "").replace("﻿", "")
    return re.sub(r"\s+", " ", text).strip()


def split_school_name(raw_name):
    text = normalize_text(raw_name)
    parens = [normalize_text(p) for p in PAREN_PATTERN.findall(text) if normalize_text(p)]
    base = normalize_text(PAREN_PATTERN.sub("", text))
    campus_raw = parens[0] if parens else ""
    return base, campus_raw


def attach_school_campus_columns(df, school_col="학교명"):
    out = df.copy()
    split = out[school_col].map(split_school_name)
    out["school_name_raw"] = out[school_col].map(normalize_text).astype("string")
    out["school_name_base_raw"] = split.map(lambda x: x[0]).astype("string")
    out["school_name_std"] = out["school_name_base_raw"]
    out["campus_name_raw"] = split.map(lambda x: x[1]).astype("string")
    out["campus_name_std"] = out["campus_name_raw"].where(out["campus_name_raw"].str.len() > 0, pd.NA).astype("category")
    return out


grade_2023 = attach_school_campus_columns(grade_2023, "학교명")
employment_2023 = attach_school_campus_columns(employment_2023, "학교명")

print(grade_2023[["학교명", "school_name_raw", "school_name_std", "campus_name_raw"]].drop_duplicates().head(8))


              학교명 school_name_raw school_name_std campus_name_raw
0       가야대학교(김해)       가야대학교(김해)           가야대학교              김해
13          가천대학교           가천대학교           가천대학교                
90       가톨릭관동대학교        가톨릭관동대학교        가톨릭관동대학교                
149     가톨릭꽃동네대학교       가톨릭꽃동네대학교       가톨릭꽃동네대학교                
155        가톨릭대학교          가톨릭대학교          가톨릭대학교                
197  가톨릭대학교_제2캠퍼스    가톨릭대학교_제2캠퍼스    가톨릭대학교_제2캠퍼스                
200  가톨릭대학교_제3캠퍼스    가톨릭대학교_제3캠퍼스    가톨릭대학교_제3캠퍼스                
201      감리교신학대학교        감리교신학대학교        감리교신학대학교                


## 4. 두 원천 병합 — outer join으로 spine 유지 (행 삭제 금지, 매칭 품질은 플래그로)

블루프린트 §3.1 원칙("매칭 실패 때문에 spine 행을 삭제하지 않는다")을 그대로 따른다.
`(school_name_std, 학과_계열)` 키로 outer merge하고 `in_grade_source`/`in_employment_source`
플래그를 남긴다.

In [10]:
MERGE_KEYS = ["school_name_std", "학과_계열"]

grade_side = grade_2023[MERGE_KEYS + ["school_name_raw", "campus_name_raw", "campus_name_std", "a_rate_pct", "cd_rate_pct", "f_rate_pct"]].copy()
grade_side["in_grade_source"] = True

emp_side = employment_2023[MERGE_KEYS + ["school_name_raw", "campus_name_raw", "campus_name_std", "employment_rate_pct", "health_employment_rate_pct", "graduates_n"]].copy()
emp_side["in_employment_source"] = True

merged_2023 = pd.merge(
    grade_side, emp_side,
    on=MERGE_KEYS, how="outer", suffixes=("_grade", "_emp"), indicator="_merge_source",
)

merged_2023["in_grade_source"] = merged_2023["in_grade_source"].fillna(False)
merged_2023["in_employment_source"] = merged_2023["in_employment_source"].fillna(False)

merged_2023["school_name_raw"] = merged_2023["school_name_raw_grade"].combine_first(merged_2023["school_name_raw_emp"])
merged_2023["campus_name_raw"] = merged_2023["campus_name_raw_grade"].combine_first(merged_2023["campus_name_raw_emp"])
merged_2023["campus_name_std"] = merged_2023["campus_name_std_grade"].combine_first(merged_2023["campus_name_std_emp"])

print("병합 결과 shape:", merged_2023.shape)
print(merged_2023["_merge_source"].value_counts())
merged_2023[MERGE_KEYS + ["a_rate_pct", "cd_rate_pct", "f_rate_pct", "employment_rate_pct", "in_grade_source", "in_employment_source"]].head(8)


병합 결과 shape: (11287, 20)
_merge_source
both          6299
left_only     2927
right_only    2061
Name: count, dtype: int64


,school_name_std,학과_계열,a_rate_pct,cd_rate_pct,f_rate_pct,employment_rate_pct,in_grade_source,in_employment_source
0,LH토지주택대학교,<NA>,NaN,NaN,NaN,100.0,False,True
1,가야대학교,간호,31.530556,25.378143,1.348014,85.9,True,True
2,가야대학교,경영물류,28.941423,29.276878,2.970188,70.4,True,True
3,가야대학교,경찰소방,28.516540,26.675937,3.798302,31.3,True,True
4,가야대학교,귀금속주얼리,33.284600,24.642625,0.000000,57.9,True,True
5,가야대학교,물리치료,25.798374,31.027107,5.218533,88.7,True,True
6,가야대학교,방사선,31.182983,28.314552,4.722493,81.4,True,True
7,가야대학교,사회복지,38.887537,20.452166,3.434007,75.4,True,True


## 5. D02(`dept_outcomes_2024.parquet`) 32컬럼 스키마에 맞춰 최종 조립

- 2023 소스로 만들 수 없는 컬럼(`selectivity_proxy_pct`, `progression_*`, `health_employment_rate_pct`,
  `credit_forfeit_flag` 등)은 **컬럼은 유지하되 값은 결측**으로 둔다(컬럼 스키마 호환 우선, 임의 대체 금지).
- `dept_name_raw`/`dept_name_std`는 이번 배치에서는 이미 정규화된 `학과_계열`을 그대로 쓴다(한계로 기록).

In [11]:
def sha256_of_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()


grade_sha = sha256_of_file(GRADE_XLSX_PATH)
emp_sha = sha256_of_file(EMPLOYMENT_XLSX_PATH)

out = merged_2023.copy()
out = out.rename(columns={"학과_계열": "dept_name_std"})
out["dept_name_raw"] = out["dept_name_std"]
out["dept_field_raw"] = pd.NA
out["dept_field_std"] = pd.NA

out["analysis_year"] = pd.array([2023] * len(out), dtype="Int16")
out["outcome_row_id"] = [f"OC2023_{i+1:05d}" for i in range(len(out))]
out["school_name_base_raw"] = out["school_name_std"]
out["campus_seq"] = pd.NA
out["campus_branch"] = pd.NA
out["credit_forfeit_flag"] = pd.array([pd.NA] * len(out), dtype="boolean")
out["selectivity_proxy_pct"] = np.nan

for col in [
    "progression_rate_pct", "vocational_college_progression_rate_pct", "university_progression_rate_pct",
    "graduate_school_progression_rate_pct", "domestic_progression_rate_pct", "overseas_progression_rate_pct",
]:
    out[col] = np.nan

out["has_selectivity"] = pd.array([False] * len(out), dtype="boolean")
out["has_employment"] = out["employment_rate_pct"].notna().astype("boolean")
out["has_progression"] = pd.array([False] * len(out), dtype="boolean")

RATE_COLS = [
    "a_rate_pct", "cd_rate_pct", "f_rate_pct", "employment_rate_pct", "health_employment_rate_pct",
    "progression_rate_pct", "vocational_college_progression_rate_pct", "university_progression_rate_pct",
    "graduate_school_progression_rate_pct", "domestic_progression_rate_pct", "overseas_progression_rate_pct",
    "selectivity_proxy_pct",
]


def _range_flag(row):
    for c in RATE_COLS:
        v = row[c]
        if pd.notna(v) and not (0 <= v <= 100):
            return "out_of_range"
    return "ok"


out["rate_range_qa"] = out.apply(_range_flag, axis=1).astype("category")

out["source_file"] = np.where(
    out["in_grade_source"] & out["in_employment_source"],
    "전공과목 성적 분포 (대학)_2023-07-1220531571.xlsx | 2023년 학교별 학과별 고등교육기관 졸업자 취업통계_241230 (1).xlsx",
    np.where(out["in_grade_source"], "전공과목 성적 분포 (대학)_2023-07-1220531571.xlsx",
             "2023년 학교별 학과별 고등교육기관 졸업자 취업통계_241230 (1).xlsx"),
)
out["source_sha256"] = np.where(
    out["in_grade_source"] & out["in_employment_source"], f"{grade_sha}|{emp_sha}",
    np.where(out["in_grade_source"], grade_sha, emp_sha),
)

# D02와 동일 dtype으로 캐스팅
for col, dtype in D02_TARGET_DTYPES.items():
    if col not in out.columns:
        continue
    try:
        if dtype in ("string",):
            out[col] = out[col].astype("string")
        elif str(dtype).startswith("Float"):
            out[col] = pd.to_numeric(out[col], errors="coerce").astype(dtype)
        elif str(dtype).startswith("Int"):
            out[col] = pd.to_numeric(out[col], errors="coerce").astype(dtype)
        elif dtype == "boolean":
            out[col] = out[col].astype("boolean")
        elif dtype == "category":
            out[col] = out[col].astype("category")
    except Exception as e:
        print(f"[dtype 캐스팅 경고] {col} -> {dtype}: {e}")

dept_outcomes_2023 = out[[c for c in D02_TARGET_COLUMNS if c in out.columns] + ["in_grade_source", "in_employment_source", "source_reported_year_raw"]] \
    if "source_reported_year_raw" in out.columns else out[[c for c in D02_TARGET_COLUMNS if c in out.columns] + ["in_grade_source", "in_employment_source"]]

dept_outcomes_2023["source_reported_year_raw_grade"] = 2022  # 성적분포 원본 기준연도 실측값(파일 타이틀=2023, 실측 값=2022)

print("최종 shape:", dept_outcomes_2023.shape)
print("D02 대비 누락 컬럼:", set(D02_TARGET_COLUMNS) - set(dept_outcomes_2023.columns))
dept_outcomes_2023.head(5)


최종 shape: (11287, 35)
D02 대비 누락 컬럼: set()


,analysis_year,outcome_row_id,school_name_raw,school_name_base_raw,school_name_std,campus_name_raw,campus_seq,campus_branch,campus_name_std,dept_name_raw,dept_name_std,dept_field_raw,dept_field_std,credit_forfeit_flag,selectivity_proxy_pct,a_rate_pct,cd_rate_pct,f_rate_pct,employment_rate_pct,health_employment_rate_pct,progression_rate_pct,vocational_college_progression_rate_pct,university_progression_rate_pct,graduate_school_progression_rate_pct,domestic_progression_rate_pct,overseas_progression_rate_pct,has_selectivity,has_employment,has_progression,rate_range_qa,source_file,source_sha256,in_grade_source,in_employment_source,source_reported_year_raw_grade
0,2023,OC2023_00001,LH토지주택대학교,LH토지주택대학교,LH토지주택대학교,,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,100.0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,True,False,ok,2023년 학교별 학과별 고등교육기관 졸업자 취업통계_241230 (1).xlsx,dd147d525afe121b189c6595bfdd2c5031cf0d56be4d11...,False,True,2022
1,2023,OC2023_00002,가야대학교(김해),가야대학교,가야대학교,김해,<NA>,<NA>,김해,간호,간호,<NA>,<NA>,<NA>,<NA>,31.530556,25.378143,1.348014,85.900002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,True,False,ok,전공과목 성적 분포 (대학)_2023-07-1220531571.xlsx | 2023...,57987fd0bb3beaafde9e438b89323b7ffda88c613c5862...,True,True,2022
2,2023,OC2023_00003,가야대학교(김해),가야대학교,가야대학교,김해,<NA>,<NA>,김해,경영물류,경영물류,<NA>,<NA>,<NA>,<NA>,28.941423,29.276878,2.970188,70.400002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,True,False,ok,전공과목 성적 분포 (대학)_2023-07-1220531571.xlsx | 2023...,57987fd0bb3beaafde9e438b89323b7ffda88c613c5862...,True,True,2022
3,2023,OC2023_00004,가야대학교(김해),가야대학교,가야대학교,김해,<NA>,<NA>,김해,경찰소방,경찰소방,<NA>,<NA>,<NA>,<NA>,28.516541,26.675938,3.798302,31.299999,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,True,False,ok,전공과목 성적 분포 (대학)_2023-07-1220531571.xlsx | 2023...,57987fd0bb3beaafde9e438b89323b7ffda88c613c5862...,True,True,2022
4,2023,OC2023_00005,가야대학교(김해),가야대학교,가야대학교,김해,<NA>,<NA>,김해,귀금속주얼리,귀금속주얼리,<NA>,<NA>,<NA>,<NA>,33.284599,24.642626,0.0,57.900002,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,False,True,False,ok,전공과목 성적 분포 (대학)_2023-07-1220531571.xlsx | 2023...,57987fd0bb3beaafde9e438b89323b7ffda88c613c5862...,True,True,2022


## 6. 검증 — Acceptance Gate 스타일 체크 (블루프린트 §8 정신 재사용)

In [12]:
print("shape:", dept_outcomes_2023.shape)
print()
dup_key = dept_outcomes_2023.duplicated(subset=["school_name_std", "campus_name_std", "dept_name_std"]).sum()
print("spine key 중복:", dup_key)
print()
print("비율 범위 위반(rate_range_qa != ok):", (dept_outcomes_2023["rate_range_qa"] != "ok").sum())
print()
print("in_grade_source / in_employment_source 교차표:")
print(pd.crosstab(dept_outcomes_2023["in_grade_source"], dept_outcomes_2023["in_employment_source"]))
print()
print("2024 school_name_std 대비 겹치는 학교 비율:")
schools_2023 = set(dept_outcomes_2023["school_name_std"].dropna().unique())
overlap = schools_2023 & school_name_std_universe_2024
print(f"  2023 고유 학교수={len(schools_2023)}  2024 고유 학교수={len(school_name_std_universe_2024)}  교집합={len(overlap)}  "
      f"2023 대비 매칭률={len(overlap)/len(schools_2023)*100:.1f}%")
print()
print("컬럼별 결측률(%):")
print((dept_outcomes_2023[[c for c in D02_TARGET_COLUMNS if c in dept_outcomes_2023.columns]].isna().mean() * 100).round(1))


shape: (11287, 35)

spine key 중복: 0

비율 범위 위반(rate_range_qa != ok): 0

in_grade_source / in_employment_source 교차표:
in_employment_source  False  True 
in_grade_source                   
False                     0   2061
True                   2927   6299

2024 school_name_std 대비 겹치는 학교 비율:
  2023 고유 학교수=267  2024 고유 학교수=200  교집합=198  2023 대비 매칭률=74.2%

컬럼별 결측률(%):
analysis_year                                0.0
outcome_row_id                               0.0
school_name_raw                              0.0
school_name_base_raw                         0.0
school_name_std                              0.0
campus_name_raw                              0.0
campus_seq                                 100.0
campus_branch                              100.0
campus_name_std                             96.3
dept_name_raw                                1.7
dept_name_std                                1.7
dept_field_raw                             100.0
dept_field_std                             10

In [13]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["a_rate_pct", "cd_rate_pct", "employment_rate_pct"]):
    dept_outcomes_2023[col].dropna().astype(float).hist(bins=30, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.savefig(P2_4_DIR / "2023_outcome_rate_distributions.png", dpi=110)
plt.show()
print("분포 히스토그램 저장 완료")


분포 히스토그램 저장 완료


/tmp/ipykernel_257545/2208031911.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. 저장 — 2023 표본 (parquet + csv)

In [14]:
OUT_PARQUET_PATH = P2_4_DIR / "dept_outcomes_2023_sample.parquet"
OUT_CSV_PATH = P2_4_DIR / "dept_outcomes_2023_sample.csv"

dept_outcomes_2023.to_parquet(OUT_PARQUET_PATH, index=False)
dept_outcomes_2023.to_csv(OUT_CSV_PATH, index=False, encoding="utf-8-sig")

print("저장 완료:")
print(" ", OUT_PARQUET_PATH, dept_outcomes_2023.shape)
print(" ", OUT_CSV_PATH)


저장 완료:
  /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/dept_outcomes_2023_sample.parquet (11287, 35)
  /home/sieg/projects-wsl/SBS_dataScience/workbook/p2/p2_4/dept_outcomes_2023_sample.csv


## 8. 알려진 한계 (숨기지 않고 명시)

1. **`기준연도` 실측값 불일치**: 성적분포 파일은 제목이 "2023년_"이지만 실제 `기준연도` 컬럼 값은
   **2022**다. 대학알리미류 공시 특성상 발행연도-1년 학사데이터로 추정된다. 이 노트북은
   `analysis_year=2023`(시계열 슬롯, 취업통계 파일의 실제 조사기준일 2023.12.31과 정렬)으로
   배정하고, 성적분포 쪽 원본 값은 `source_reported_year_raw_grade=2022`로 별도 보존했다.
2. **`health_employment_rate_pct`(건보가입취업률) 산출 불가**: 취업통계 파일에 건강보험 직장가입자
   컬럼이 없다. 전부 결측으로 남겼다(임의 추정 금지).
3. **`selectivity_proxy_pct`, `progression_*` 계열 전부 결측**: 이번 2023 배치에는 정시입결/진학률
   원천 파일이 제공되지 않았다. 컬럼은 D02와 동일하게 유지하되 전부 NaN이다.
4. **`dept_name_raw`가 `dept_name_std`와 동일**: 학과 원문 명칭을 그룹핑 이전 단계에서 보존하지
   않았다 — 필요 시 `grade_f`/`emp_f`의 원본 학과_전공/학과명 컬럼에서 별도로 join_unique를
   추가해 재구성할 수 있다(이번 배치 범위 밖).
5. **`campus_name_std` 커버리지 제한**: 성적분포 파일은 학교명에 괄호 캠퍼스 표기가 있는 경우만
   캠퍼스가 식별된다. 취업통계 파일의 `본분교`(본교/캠퍼스/분교) 컬럼은 `campus_status_raw`로만
   보존했고 `campus_name_std`에는 아직 반영하지 않았다.
6. 이 산출물은 **표본(sample)**이며 `p2_3`의 D01~D08 파이프라인에 자동 병합되지 않는다 — 병합은
   P2-G3 담당 에이전트가 별도로 검토·승인한 뒤 진행해야 한다.
